In [4]:
import tensorflow as tf
import numpy as np
import librosa, librosa.feature

SEED = 42
SR, HOP, N_FFT, N_MELS = 16000, 160, 400, 40  # 10 ms hop, 25 ms window
CHUNK_FRAMES = 200  # 2 seconds per training example
BATCH_SIZE = 16

In [22]:
model = tf.keras.models.load_model("/home/sde_sep1_ai1/Project/Software/step1/keras_vad.keras")

TypeError: Error when deserializing class 'InputLayer' using config={'batch_shape': [None, None, 40], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_5', 'optional': False}.

Exception encountered: Unrecognized keyword arguments: ['batch_shape', 'optional']

In [ ]:
def load_and_extract(path):
    # converts to mono and resamples to 16 kHz. (y=samples)
    y, sr = librosa.load(path, sr=SR, mono=True)
    if len(y) < N_FFT:
        y = np.pad(y, (0, N_FFT - len(y)))

    # compute mel spectrogram (returns 2D array Hz & time)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, center=False)

    # convert to decibel
    log_mel = librosa.power_to_db(mel, ref=np.max)
    return log_mel.T.astype(np.float32)

In [13]:
wav_path = "/home/sde_sep1_ai1/Project/Software/data/audio/Female/TMIT/SA1.wav"
features = load_and_extract(wav_path)

X = features[np.newaxis, ...]
probabilities = model.predict(X, verbose=0)[0, :, 0]
predictions = (probabilities >= 0.5).astype(int)